In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os
os.chdir(r"C:\Users\A_PORTEU\python_projects\lib_climate_alignment")

In [ ]:
from _02_targets import plot_sector_region_fraction_heatmap_with_text

In [3]:
first_year_available, last_year_available = 2019, 2024

In [4]:
df = pd.read_parquet(
    f"data/intermediate_data/df_merged_all_infos_{first_year_available}_{last_year_available}.parquet"
)
df_em_after_outliers_treatment = pd.read_parquet(
    "data/intermediate_data/df_em_after_outliers_treatment.parquet"
)
df_targets = pd.read_excel("data/intermediate_data/df_eq_target_ambitions.xlsx")


In [5]:
all_isin_index = df[df['is_relevant_scopes']==True].set_index('isin').index
all_isin_rel_scope =  df[df['is_relevant_scopes']==True].set_index(['isin', 'scope']).index

regions = list(df["region_0"].unique())
list_high_impact_sector = list(df["high_impact_sector"].dropna().sort_values().unique())
delta_years_str = [
    f"{int(y)}-{y+1}" for y in range(first_year_available, last_year_available)
]
threshold = 0
target_needed = "is_target_sbt_or_ambitious_or_commited"

In [6]:
# eq with average rate inferior to threshold 
eq_all_scope_pass = df_em_after_outliers_treatment.set_index(['isin', 'scope'])[delta_years_str].mean(axis=1) < threshold
eq_rel_scp_pass = (eq_all_scope_pass
                   .loc[all_isin_rel_scope.intersection(eq_all_scope_pass.index)]
                   .reset_index().set_index('isin')[0])
eq_rel_scp_pass.name = 'below_threshold'

# eq with specified target ambition
eq_target_pass = df_targets[df_targets[target_needed]==True][['isin', target_needed]].set_index('isin')
eq_target_pass = eq_target_pass.loc[all_isin_index.intersection(eq_target_pass.index)].astype(bool)

# merge
df_all_info_for_alignment = pd.concat([df[df['is_relevant_scopes']==True].set_index('isin')[
    ['scope', 'high_impact_sector', 'region_0', 'r_s_mc_weight']], eq_rel_scp_pass, eq_target_pass], axis=1)
df_all_info_for_alignment

,scope,high_impact_sector,region_0,r_s_mc_weight,below_threshold,is_target_sbt_or_ambitious_or_commited
isin,,,,,,
US88025U1097,s12,Industrials,Dev America US,0.000106,NaN,NaN
KYG6S54B1005,s12,Consumer goods & services,Emg Asia-Pac CN,0.015597,NaN,NaN
KYG6757R1056,s12,non-HIMS,Dev America US,0.000006,NaN,NaN
US3369011032,s3_d,Banking,Dev America US,0.000289,False,NaN
US3205511047,s12,non-HIMS,Dev America US,0.000005,NaN,NaN
...,...,...,...,...,...,...
IE00BDVJJQ56,s12,Industrials,Dev Europe GB,0.041902,True,True
AU000000NHF0,s12,non-HIMS,Dev Asia-Pac Other,0.003208,NaN,True
DE0007500001,s12,Steel,Dev Europe EU,0.107445,False,True


In [18]:
# to erase
eq_all_scope_pass = df_em_after_outliers_treatment.set_index(['isin', 'scope'])[delta_years_str].mean(axis=1) < threshold
eq_all_scope_pass.reindex(all_isin_rel_scope).reset_index().set_index("isin")[0]
eq_all_scope_pass.reindex(all_isin_rel_scope).reset_index(level="scope", drop=True)

isin
US88025U1097      NaN
KYG6S54B1005      NaN
KYG6757R1056      NaN
US3369011032    False
US3205511047      NaN
                ...  
IE00BDVJJQ56     True
AU000000NHF0      NaN
DE0007500001    False
NL0010696654      NaN
DK0060094928    False
Length: 7574, dtype: object

In [7]:
# compute who is aligned
df_all_info_for_alignment['aligned'] = (df_all_info_for_alignment[['below_threshold', 'is_target_sbt_or_ambitious_or_commited']]
                                        .astype('boolean').fillna(False).all(axis=1))
df_all_info_for_alignment['challenger'] = (df_all_info_for_alignment[['below_threshold', 'is_target_sbt_or_ambitious_or_commited']]
                                        .astype('boolean').fillna(False).any(axis=1))

In [8]:
plot_sector_region_fraction_heatmap_with_text(
    df_all_info_for_alignment[df_all_info_for_alignment['aligned']].reset_index(), 
    df_all_info_for_alignment.reset_index(),
    "output/fraction_eq_aligned_heatmap.png",
    "Nbr aligned on total number by region and sectors",
    reverse_colors=False, 
    )

plot_sector_region_fraction_heatmap_with_text(
    df_all_info_for_alignment[df_all_info_for_alignment['challenger']].reset_index(), 
    df_all_info_for_alignment.reset_index(),
    "output/fraction_eq_challenger_heatmap.png",
    "Nbr challenger on total number by region and sectors",
    reverse_colors=False, 
    )

# TO ERASE TEST

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# ----------------------------
# Run Streamlit:
# streamlit run analysis/st_aligned_challenger.py
# ----------------------------

first_year_available, last_year_available = 2019, 2024

df = pd.read_parquet(
    f"data/intermediate_data/df_merged_all_infos_{first_year_available}_{last_year_available}.parquet"
)
df_em_after_outliers_treatment = pd.read_parquet(
    "data/intermediate_data/df_em_after_outliers_treatment.parquet"
)
df_targets = pd.read_excel("data/intermediate_data/df_eq_target_ambitions.xlsx")

all_isin_index = df[df["is_relevant_scopes"] == True].set_index("isin").index
all_isin_rel_scope = (
    df[df["is_relevant_scopes"] == True].set_index(["isin", "scope"]).index
)

regions = list(df["region_0"].unique())
list_high_impact_sector = list(df["high_impact_sector"].dropna().sort_values().unique())
delta_years_str = [
    f"{int(y)}-{y+1}" for y in range(first_year_available, last_year_available)
]

# ----------------------------
# Alignment parameters
# ----------------------------

threshold = 0
target_ambitions_labels_selected = ["Approved SBT"]

dict_mapping_targets_ctgry = {
    "Ambitious Target": "is_ambitious_target",
    "Approved SBT": "is_approved_sbt_target",
    "Committed SBT": "is_committed_sbt_target",
    "Non-Ambitious Target": "is_target_non_ambitious",
    "No Target": "is_no_target",
}

target_ambitions_selected = [
    dict_mapping_targets_ctgry[lbl] for lbl in target_ambitions_labels_selected
]
# ----------------------------
# Eq with average rate inferior to threshold
# ----------------------------

eq_all_scope_pass = (
    df_em_after_outliers_treatment.set_index(["isin", "scope"])[delta_years_str].mean(
        axis=1
    )
    < threshold
)
eq_rel_scp_pass = eq_all_scope_pass.reindex(all_isin_rel_scope).reset_index(
    level="scope", drop=True
)
eq_rel_scp_pass.name = "below_threshold"

# ----------------------------
# Eq with specified target ambition
# ----------------------------

eq_target_pass = df_targets.set_index("isin").reindex(all_isin_index)[target_ambitions_selected].any(axis=1)

# ----------------------------
# Compute aligned and challenger
# ----------------------------
df_all_info_for_alignment = pd.concat(
    [
        df[df["is_relevant_scopes"] == True].set_index("isin")[
            ["scope", "high_impact_sector", "region_0", "r_s_mc_weight"]
        ],
        eq_rel_scp_pass,
        eq_target_pass,
    ],
    axis=1,
)

df_all_info_for_alignment["Aligned"] = (
    df_all_info_for_alignment[["below_threshold", "has_target_needed"]]
    .astype("boolean")
    .fillna(False)
    .all(axis=1)
)
df_all_info_for_alignment["Challenger"] = (
    df_all_info_for_alignment[["below_threshold", "has_target_needed"]]
    .astype("boolean")
    .fillna(False)
    .any(axis=1)
)

# ----------------------------
# Create table
# ----------------------------


def create_alignment_heatmap(df, aligned_or_challenger):
    df_color = df.pivot_table(
        index="high_impact_sector",
        columns="region_0",
        values=aligned_or_challenger,
        aggfunc="mean",
    ).reindex(index=list_high_impact_sector, columns=regions)

    num = df.pivot_table(
        index="high_impact_sector",
        columns="region_0",
        values=aligned_or_challenger,
        aggfunc=lambda x: x.fillna(False).sum(),
    ).reindex(index=list_high_impact_sector, columns=regions)

    den = df.pivot_table(
        index="high_impact_sector",
        columns="region_0",
        values=aligned_or_challenger,
        aggfunc="count",
    ).reindex(index=list_high_impact_sector, columns=regions)

    df_frac = (
        num.fillna(0).astype(int).astype(str)
        + "/"
        + den.fillna(0).astype(int).astype(str)
    )

    return df_color, df_frac


df_color, df_frac = create_alignment_heatmap(df_all_info_for_alignment, "Aligned")


In [26]:
df_all_info_for_alignment

,scope,high_impact_sector,region_0,r_s_mc_weight,below_threshold,has_target_needed,Aligned,Challenger
isin,,,,,,,,
US88025U1097,s12,Industrials,Dev America US,0.000106,NaN,True,False,True
KYG6S54B1005,s12,Consumer goods & services,Emg Asia-Pac CN,0.015597,NaN,True,False,True
KYG6757R1056,s12,non-HIMS,Dev America US,0.000006,NaN,True,False,True
US3369011032,s3_d,Banking,Dev America US,0.000289,False,True,False,True
US3205511047,s12,non-HIMS,Dev America US,0.000005,NaN,True,False,True
...,...,...,...,...,...,...,...,...
IE00BDVJJQ56,s12,Industrials,Dev Europe GB,0.041902,True,True,True,True
AU000000NHF0,s12,non-HIMS,Dev Asia-Pac Other,0.003208,NaN,True,False,True
DE0007500001,s12,Steel,Dev Europe EU,0.107445,False,True,False,True


In [21]:
df_frac

region_0,Dev America US,Emg Asia-Pac CN,Emg Asia-Pac IN,Dev Europe GB,Dev America CA,Dev Europe Other,Dev Europe EU,Dev Asia-Pac JP,Emg Asia-Pac Other,Emg EMEA,Dev Asia-Pac Other,Emg America
high_impact_sector,,,,,,,,,,,,
"Agriculture, forestry and fishing",3/8,0/2,0/0,1/2,0/0,0/4,0/0,0/0,1/4,0/0,0/1,0/0
Airlines,3/23,3/6,1/3,0/4,0/3,0/3,1/5,1/4,1/3,0/2,1/3,0/3
Aluminium,3/4,6/10,1/2,0/0,0/0,2/2,1/1,2/3,0/1,1/1,0/0,0/0
Automobiles,9/62,7/41,0/11,0/4,0/4,2/8,3/23,5/25,1/6,0/0,2/8,0/0
Banking,22/397,17/38,1/39,3/33,0/13,2/46,1/45,11/59,0/44,2/55,2/21,2/22
Cement,1/1,2/2,0/4,0/0,0/0,1/2,2/3,1/1,2/4,0/0,0/0,2/2
Chemicals,97/433,15/77,11/24,11/31,4/15,14/50,30/51,49/61,7/21,3/4,8/14,0/3
Coal mining,5/10,0/6,0/1,0/0,0/0,0/0,0/0,0/0,0/4,0/0,2/4,0/0
Consumer goods & services,80/169,4/17,0/7,19/27,4/12,9/20,16/23,22/49,5/13,5/10,9/22,6/9


In [ ]:
eq_target_pass = df_targets.set_index("isin").reindex(all_isin_index)[target_ambitions_selected].any(axis=1)


In [ ]:
df_targets.set_index("isin").reindex(all_isin_index)[target_ambitions_selected].any(axis=1)

isin
US88025U1097    False
KYG6S54B1005    False
KYG6757R1056    False
US3369011032    False
US3205511047    False
                ...  
IE00BDVJJQ56    False
AU000000NHF0    False
DE0007500001     True
NL0010696654    False
DK0060094928     True
Length: 7574, dtype: bool